# October Gate 2 - Constant Baselines (`OCT-SPLIT-001`)

This notebook fits the two required constant baselines from the modeling contract and scores them
on the validation partition of the frozen split:

- **training-mean constant** (the required baseline): the mean `loss` of the training partition,
  predicted for every validation claim;
- **training-median constant** (comparator): the median `loss` of the training partition, reported
  beside the mean baseline, not in place of it.

Both are scored with MAE in original loss units. Every later model must beat the mean baseline to
show it learned anything from the predictors.

Only training `loss` is used to fit the constants, and only validation `loss` to score them.
Test-partition rows are dropped before `loss` is touched, so no test target is examined. The mean
or median of the full file, and the error of predicting it, describe the data rather than held-out
performance and are not reported here.

The notebook writes `baseline_results.csv`, `baseline_checks.csv`, and `baseline_run_receipt.json`
next to itself. It stops before writing anything if a check fails.

**Colab:** mount Drive and set `ROOT_OVERRIDE` in the next cell to the folder that contains
`data/allstate_claims_data.csv`.

In [1]:
import hashlib
import json
import math
import platform
import subprocess
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics import mean_absolute_error


def find_repo_root(start):
    for path in [start, *start.parents]:
        if (path / "data/allstate_claims_data.csv").exists():
            return path
    raise FileNotFoundError(
        "Could not find data/allstate_claims_data.csv above the working directory. "
        "Set ROOT to the repository folder by hand."
    )


# Colab: set this to your Drive copy of the repository, e.g. "/content/drive/MyDrive/<your-folder>"
ROOT_OVERRIDE = None
ROOT = Path(ROOT_OVERRIDE) if ROOT_OVERRIDE else find_repo_root(Path.cwd().resolve())
SOURCE_PATH = ROOT / "data/allstate_claims_data.csv"
GATE1_DIR = ROOT / "notebooks/final-deliverables/October/Gate 1"
MANIFEST_PATH = GATE1_DIR / "partition_manifest.csv"
RUN_MANIFEST_PATH = GATE1_DIR / "modeling_run_manifest.json"
OUT_DIR = ROOT / "notebooks/final-deliverables/October/Gate 2"
NOTEBOOK_PATH = OUT_DIR / "constant_baselines.ipynb"
RESULTS_PATH = OUT_DIR / "baseline_results.csv"
PREDICTIONS_PATH = OUT_DIR / "baseline_predictions.csv"
CHECKS_PATH = OUT_DIR / "baseline_checks.csv"
RECEIPT_PATH = OUT_DIR / "baseline_run_receipt.json"

# The split, its manifest hash, and the source hash are registered once, in the Gate 1 run manifest.
run_manifest = json.loads(RUN_MANIFEST_PATH.read_text(encoding="utf-8"))
SPLIT = run_manifest["split"]
print(f"Modeling data: {run_manifest['modeling_data_version']}  split: {SPLIT['version']} ({SPLIT['status']})")

Modeling data: OCT-MD-001  split: OCT-SPLIT-001 (frozen)


## Confirm the source and the frozen split

The source file and the partition manifest must be the exact files registered in Gate 1.

In [2]:
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


source_sha256 = sha256_file(SOURCE_PATH)
manifest_sha256 = sha256_file(MANIFEST_PATH)
print(f"Source SHA-256 matches run manifest: {source_sha256 == run_manifest['source_sha256']}")
print(f"Manifest SHA-256 matches {SPLIT['version']}: {manifest_sha256 == SPLIT['manifest_sha256']}")

Source SHA-256 matches run manifest: True
Manifest SHA-256 matches OCT-SPLIT-001: True


## Keep train and validation claims only

Test rows are removed from the manifest first; `loss` is joined by `id` to the remaining claims.

In [3]:
manifest = pd.read_csv(MANIFEST_PATH, usecols=["id", "partition"])
manifest = manifest[manifest["partition"].isin(["train", "validation"])]
source = pd.read_csv(SOURCE_PATH, usecols=["id", "loss"])
claims = manifest.merge(source, on="id", how="left", validate="one_to_one")
del source

train = claims[claims["partition"] == "train"]
validation = claims[claims["partition"] == "validation"].sort_values("id", ignore_index=True)
print(f"Train claims: {len(train):,}  validation claims: {len(validation):,}")

Train claims: 131,822  validation claims: 28,248


## Fit the constants on training `loss` only

In [4]:
constants = {
    "training_mean_constant": float(train["loss"].mean()),
    "training_median_constant": float(train["loss"].median()),
}
for model, constant in constants.items():
    print(f"{model}: {constant:,.4f}")

training_mean_constant: 3,038.8403
training_median_constant: 2,115.3900


## Predict one value for every validation claim and score it

Predictions are keyed by claim `id` and joined to the true `loss` by `id`, so the score does not
depend on row order.

In [5]:
predictions = pd.DataFrame({"id": validation["id"].to_numpy()})
for model, constant in constants.items():
    predictions[model] = constant

scored = predictions.merge(validation[["id", "loss"]], on="id", how="left", validate="one_to_one")
mae = {model: float(mean_absolute_error(scored["loss"], scored[model])) for model in constants}
for model, value in mae.items():
    print(f"{model}: validation MAE {value:,.4f}")

training_mean_constant: validation MAE 1,962.9187
training_median_constant: validation MAE 1,805.0864


## Checks

Each check records its expected and observed value. Any failed check stops the notebook before
results are written. B7 recomputes each MAE with a plain Python loop over (`id`, `loss`) pairs,
independently of the pandas/scikit-learn path above.

In [6]:
checks = []


def record(check_id, description, expected, observed, passed):
    checks.append({
        "check_id": check_id,
        "description": description,
        "expected": str(expected),
        "observed": str(observed),
        "status": "pass" if passed else "fail",
    })


record("B1", "Source SHA-256 matches the Gate 1 run manifest",
       run_manifest["source_sha256"], source_sha256, source_sha256 == run_manifest["source_sha256"])

record("B2", f"Partition manifest SHA-256 matches frozen {SPLIT['version']}",
       SPLIT["manifest_sha256"], manifest_sha256, manifest_sha256 == SPLIT["manifest_sha256"])

counts = {"train": len(train), "validation": len(validation)}
expected_counts = {"train": SPLIT["train_claims"], "validation": SPLIT["validation_claims"]}
record("B3", "Train and validation claim counts match the frozen split",
       expected_counts, counts, counts == expected_counts)

one_each = (
    len(predictions) == len(validation)
    and predictions["id"].is_unique
    and set(predictions["id"]) == set(validation["id"])
)
record("B4", "Exactly one prediction per validation claim",
       f"{len(validation)} unique validation ids",
       f"{len(predictions)} rows, {predictions['id'].nunique()} unique, same set: "
       f"{set(predictions['id']) == set(validation['id'])}",
       one_each)

aligned = (
    scored["id"].equals(validation["id"])
    and not scored["loss"].isna().any()
    and np.array_equal(scored["loss"].to_numpy(), validation["loss"].to_numpy())
)
record("B5", "Each prediction is joined to its own claim's loss by id",
       "scored ids and losses equal validation ids and losses", f"aligned: {aligned}", aligned)

finite = bool(np.isfinite(predictions[list(constants)].to_numpy()).all())
record("B6", "All predictions are finite", True, finite, finite)

truth = dict(zip(validation["id"].tolist(), validation["loss"].tolist()))
independent = {
    model: math.fsum(abs(truth[claim_id] - float(prediction))
                     for claim_id, prediction in zip(predictions["id"].tolist(), predictions[model].tolist()))
    / len(truth)
    for model in constants
}
mae_ok = all(abs(mae[model] - independent[model]) <= 1e-9 * max(1.0, independent[model]) for model in constants)
record("B7", "MAE matches an independent plain-Python calculation",
       {m: round(v, 6) for m, v in independent.items()}, {m: round(v, 6) for m, v in mae.items()}, mae_ok)

checks = pd.DataFrame(checks)
print(checks[["check_id", "description", "status"]].to_string(index=False))
failed = checks.loc[checks["status"] == "fail", "check_id"].tolist()
assert not failed, f"Checks failed: {failed}; nothing was written"

check_id                                              description status
      B1           Source SHA-256 matches the Gate 1 run manifest   pass
      B2  Partition manifest SHA-256 matches frozen OCT-SPLIT-001   pass
      B3 Train and validation claim counts match the frozen split   pass
      B4              Exactly one prediction per validation claim   pass
      B5  Each prediction is joined to its own claim's loss by id   pass
      B6                               All predictions are finite   pass
      B7      MAE matches an independent plain-Python calculation   pass


## Results

Both MAEs are in original loss units on the same 28,248 validation claims. The mean constant is
the required baseline; the median constant is a comparator reported beside it.

In [7]:
roles = {"training_mean_constant": "required baseline", "training_median_constant": "comparator"}
results = pd.DataFrame([
    {
        "model": model,
        "role": roles[model],
        "fitted_on": "train",
        "evaluated_on": "validation",
        "fitted_constant": constants[model],
        "train_claims": len(train),
        "validation_claims": len(validation),
        "validation_mae": mae[model],
    }
    for model in constants
])

OUT_DIR.mkdir(parents=True, exist_ok=True)

results.to_csv(RESULTS_PATH, index=False, lineterminator="\n")
predictions.to_csv(PREDICTIONS_PATH, index=False, lineterminator="\n")
checks.to_csv(CHECKS_PATH, index=False, lineterminator="\n")

results

,model,role,fitted_on,evaluated_on,fitted_constant,train_claims,validation_claims,validation_mae
0,training_mean_constant,required baseline,train,validation,3038.840255,131822,28248,1962.918654
1,training_median_constant,comparator,train,validation,2115.390000,131822,28248,1805.086353


## Run receipt

`notebook_code_sha256` hashes this notebook's code cells, so it identifies the code that produced
these results even before they are committed. `git_commit` is the commit the run started from, and
`git_dirty` records whether the working tree had uncommitted changes at the time.

In [8]:
def git(*args):
    try:
        return subprocess.run(["git", *args], cwd=ROOT, capture_output=True, text=True, check=True).stdout.strip()
    except (OSError, subprocess.CalledProcessError):
        return None  # e.g. a Colab copy that is not a git checkout


notebook = json.loads(NOTEBOOK_PATH.read_text(encoding="utf-8"))
notebook_code = "\n\n".join("".join(cell["source"]) for cell in notebook["cells"] if cell["cell_type"] == "code")
notebook_code_sha256 = hashlib.sha256(notebook_code.encode("utf-8")).hexdigest()
git_status = git("status", "--porcelain")

run_utc = datetime.now(timezone.utc)
run_key = hashlib.sha256(f"{notebook_code_sha256}{manifest_sha256}{source_sha256}".encode("utf-8")).hexdigest()
receipt = {
    "run_id": f"OCT-BASE-{run_utc:%Y%m%dT%H%M%SZ}-{run_key[:8]}",
    "run_utc": run_utc.isoformat(timespec="seconds"),
    "modeling_data_version": run_manifest["modeling_data_version"],
    "split_version": SPLIT["version"],
    "manifest_path": str(MANIFEST_PATH.relative_to(ROOT)),
    "manifest_sha256": manifest_sha256,
    "source_path": str(SOURCE_PATH.relative_to(ROOT)),
    "source_sha256": source_sha256,
    "notebook_path": str(NOTEBOOK_PATH.relative_to(ROOT)),
    "notebook_code_sha256": notebook_code_sha256,
    "git_commit": git("rev-parse", "HEAD"),
    "git_dirty": None if git_status is None else bool(git_status),
    "library_versions": {
        "python": platform.python_version(),
        "scikit-learn": sklearn.__version__,
        "pandas": pd.__version__,
        "numpy": np.__version__,
    },
    "metric": "MAE",
    "metric_units": "original loss units",
    "results": results.to_dict(orient="records"),
    "checks": dict(zip(checks["check_id"], checks["status"])),
}
RECEIPT_PATH.write_text(json.dumps(receipt, indent=2) + "\n", encoding="utf-8")
print(json.dumps(receipt, indent=2))

{
  "run_id": "OCT-BASE-20261021T040859Z-1d82244e",
  "run_utc": "2026-10-21T04:08:59+00:00",
  "modeling_data_version": "OCT-MD-001",
  "split_version": "OCT-SPLIT-001",
  "manifest_path": "notebooks\\final-deliverables\\October\\Gate 1\\partition_manifest.csv",
  "manifest_sha256": "de10b0e17ad56174a7b126352a0a41b3b3d477bbddd65f37b87a6235f582a64b",
  "source_path": "data\\allstate_claims_data.csv",
  "source_sha256": "74037cb248a1064e4d578692a4f4e5d8492ed1b2033daf643496e1b68b14ae03",
  "notebook_path": "notebooks\\final-deliverables\\October\\Gate 2\\constant_baselines.ipynb",
  "notebook_code_sha256": "921dff4e504a60ab134c02d288e6a69b6f9a7686df60e1fdeb1ff927bdf5d4ee",
  "git_commit": "1812832dcdffde7b450872e27c90e9201c9ac0ef",
  "git_dirty": true,
  "library_versions": {
    "python": "3.12.8",
    "scikit-learn": "1.5.1",
    "pandas": "2.2.2",
    "numpy": "1.26.4"
  },
  "metric": "MAE",
  "metric_units": "original loss units",
  "results": [
    {
      "model": "training_mean_c